<a href="https://colab.research.google.com/github/AndresAvella09/Proyecto_Multimodal/blob/main/notebooks/C3_ogm_ge_tabla_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# C3 — OGM-GE, tabla final y análisis de fallos

**Qué dejaron el C1 y el C2:**
- Repesar las modalidades **al combinar** (compuerta por entropía, con dev limpio o con degradación sintética)
  no mejora con las películas recientes.
- El texto de las recientes se equivoca **con seguridad** (entropía igual, exceso positivo). La degradación por
  mezcla con otra película reproduce ese patrón, y ahí el término de entropía tampoco ayuda.
- No aparece *greedy modality* clásica, ni siquiera con el 2% del train: la fusión siempre le gana a la mejor rama.

**Lo que falta probar:** la solución estándar del área, que actúa **en el entrenamiento**.
**OGM-GE** (*On-the-fly Gradient Modulation with Generalization Enhancement*, Peng et al., CVPR 2022):

1. En cada lote se mide cuánto aporta cada modalidad a la fusión. Como la capa de salida es lineal sobre la
   concatenación, el logit se separa en una parte por modalidad, $z_m = W_m h_m + b/2$. El aporte $s_m$ es la
   probabilidad media que $z_m$ le da a los géneros correctos (la versión multietiqueta del artículo).
2. Si una modalidad domina ($r = s_{\text{img}} / s_{\text{txt}}$ lejos de 1), a **su** proyección se le frena el
   gradiente: $k = 1 - \tanh\big(\alpha \cdot (r_{\text{dom}} - 1)\big)$. Así la otra alcanza a aprender.
3. **GE:** se suma ruido gaussiano al gradiente de las proyecciones, con la misma desviación que el gradiente,
   para compensar que el frenado reduce la exploración.

La idea es que la imagen aprenda más **dentro** de la fusión. Si eso hace a la fusión menos dependiente del
texto, debería resistir mejor cuando el texto falla (recientes).

`α` se elige en dev con la semilla 0. Se entrenan dos versiones con 5 semillas: `ogm` (sin ruido) y `ogm_ge`.
Al final, **una sola tabla** con todos los métodos del proyecto en los cuatro grupos del 08.

### Criterios fijados antes de ver los resultados

1. **Equilibrio:** el índice de dependencia de la imagen de `ogm_ge` es mayor que el de la fusión ingenua (test,
   IC sobre semillas sin el 0).
2. **No inferioridad:** en `test_mmimdb`, el IC 95% bootstrap de AP(`ogm_ge`) − AP(ingenua) queda por encima de
   −0.5 puntos.
3. **Mejora con películas nuevas:** en `recientes_tmdb`, ese IC queda por encima de 0.
4. **La ventaja crece con la época:** ganancia en `recientes_tmdb` mayor que en `control_tmdb` (diferencia en
   diferencias, IC sin el 0).

Si el 1 se cumple y el 3 no, hay más equilibrio pero eso no sirve contra el desplazamiento de época.

In [ ]:
import os, sys, json, subprocess
EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

REPO = '/content/Proyecto_Multimodal'
if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/AndresAvella09/Proyecto_Multimodal.git', REPO], check=True)
sys.path.insert(0, REPO)

import time
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from src.datos import cargar, aplicar_estandarizacion
from src.modelos import cargar_artefactos, predecir_logits, entrenar_ogm
from src.metricas import ap_por_clase_rapido, ic95, bce_por_etiqueta, ajustar_temperatura
from src.fusion import entropia_binaria, combinar, cargar_compuertas

DATA = '/content/drive/MyDrive/proyecto_multimodal/data'
EMB = '/content/drive/MyDrive/proyecto_multimodal/embeddings'
RES = '/content/drive/MyDrive/proyecto_multimodal/results'

SEED, N_BOOT = 42, 1000
ALFAS = [0.1, 0.3, 0.5, 1.0]
MARGEN_NO_INFERIORIDAD = 0.005
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3, 'figure.dpi': 110})
GRUPOS = ['test_mmimdb', 'control_mmimdb', 'control_tmdb', 'recientes_tmdb']
METODOS = ['ingenua', 'tardia', 'compuerta', 'tardia3', 'compuerta3', 'tardia_aum', 'compuerta_aum', 'ogm', 'ogm_ge']
ORIGEN = {'ingenua': '06', 'tardia': 'C1', 'compuerta': 'C1', 'tardia3': 'C1', 'compuerta3': 'C1',
          'tardia_aum': 'C2', 'compuerta_aum': 'C2', 'ogm': 'C3', 'ogm_ge': 'C3'}
COL = {'ingenua': '#10b981', 'tardia': '#a3a3a3', 'compuerta': '#7c3aed', 'tardia3': '#525252', 'compuerta3': '#db2777',
       'tardia_aum': '#78716c', 'compuerta_aum': '#f472b6', 'ogm': '#0ea5e9', 'ogm_ge': '#1d4ed8'}

## 1. Datos, modelos del 06 y compuertas del C1 y C2

In [ ]:
art = cargar_artefactos(f'{RES}/exp2_hoy_artefactos.pt', device)
assert art['etiquetas'] == 'hoy'
MODELOS, TEMP, CONFIG = dict(art['modelos']), dict(art['temperaturas']), art['config']
SEMILLAS = sorted({s for _, s in MODELOS})
G_ENT = art['generos_entrenamiento']
MODS = ['img', 'txt']

c1, c2 = cargar_compuertas(f'{RES}/exp4_compuertas.pt'), cargar_compuertas(f'{RES}/exp5_compuertas_aum.pt')
assert c1['generos'] == c2['generos']
GENEROS = c1['generos']
COLS = np.array([G_ENT.index(g) for g in GENEROS])
COMPUERTAS = {**c1['compuertas'], **c2['compuertas']}
ENTRADAS = {**c1['entradas'], **c2['entradas']}

X_mm, Y_mm, idx_mm, _, _ = cargar(DATA, EMB, etiquetas='hoy')
Xz_mm = aplicar_estandarizacion(X_mm, art['stats_estandarizacion'])
tm = pd.read_parquet(f'{DATA}/tmdb_meta.parquet')
Xz_tm = aplicar_estandarizacion({'img': np.load(f'{EMB}/tmdb_clip_img.npy'), 'txt': np.load(f'{EMB}/tmdb_distilbert_txt.npy')},
                                art['stats_estandarizacion'])
Y_tm = tm[[f'hoy_{g}' for g in G_ENT]].values.astype(np.float32)
i_ctrl, i_rec = np.where(tm['grupo'] == 'control')[0], np.where(tm['grupo'] == 'reciente')[0]
filas_ctrl = tm['fila_hdf5'].values[i_ctrl]

conjuntos = {
    'dev_mmimdb':     ({m: Xz_mm[m][idx_mm['dev']] for m in MODS}, Y_mm[idx_mm['dev']]),
    'test_mmimdb':    ({m: Xz_mm[m][idx_mm['test']] for m in MODS}, Y_mm[idx_mm['test']]),
    'control_mmimdb': ({m: Xz_mm[m][filas_ctrl] for m in MODS}, Y_mm[filas_ctrl]),
    'control_tmdb':   ({m: Xz_tm[m][i_ctrl] for m in MODS}, Y_tm[i_ctrl]),
    'recientes_tmdb': ({m: Xz_tm[m][i_rec] for m in MODS}, Y_tm[i_rec]),
}
YC = {c: y[:, COLS] for c, (_, y) in conjuntos.items()}
print(f"{len(MODELOS)} modelos del 06 | {len(COMPUERTAS)} compuertas (C1 + C2) | {len(GENEROS)} géneros | {device}")
for c in conjuntos:
    print(f"  {c:15s} n={len(YC[c]):5d}")

## 2. Entrenamiento con OGM-GE

Mismo protocolo y configuración que la fusión ingenua del 06 (AdamW, early stopping en el AP de dev). Primero se
elige `α` con la semilla 0; después se entrenan las 5 semillas con y sin GE.

In [ ]:
CFG = {k: CONFIG[k] for k in ['dim_proy', 'dropout', 'lr', 'weight_decay', 'batch', 'max_epocas', 'paciencia']}
tens = lambda X, apagar=None: [torch.zeros(len(X[m]), X[m].shape[1], device=device) if m == apagar
                               else torch.from_numpy(X[m]).to(device) for m in MODS]
Xtr = {m: Xz_mm[m][idx_mm['train']] for m in MODS}
Ytr = torch.from_numpy(Y_mm[idx_mm['train']]).to(device)
Xdev, Ydev_all = conjuntos['dev_mmimdb'][0], Y_mm[idx_mm['dev']]

def ap_macro(Y, z):
    return float(np.nanmean(ap_por_clase_rapido(Y, z)))

t0 = time.time()
busqueda, HIST = [], {}
for a in ALFAS:
    modelo, h = entrenar_ogm(tens(Xtr), Ytr, tens(Xdev), Ydev_all, CFG, SEMILLAS[0], a, ge=True)
    HIST[a] = h
    busqueda.append({'alfa': a, 'ap_dev': ap_macro(YC['dev_mmimdb'], predecir_logits(modelo, tens(Xdev))[:, COLS]),
                     'mejor_epoca': h['mejor_epoca'], 'razon_final': h['razon'][h['mejor_epoca'] - 1]})
busqueda = pd.DataFrame(busqueda)
ALFA = float(busqueda.loc[busqueda['ap_dev'].idxmax(), 'alfa'])
print(busqueda.round(4).to_string(index=False))
print(f"\nα elegido en dev: {ALFA} | {time.time() - t0:.0f} s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for a, h in HIST.items():
    axes[0].plot(h['razon'], label=f'α = {a}', lw=2 if a == ALFA else 1)
    axes[1].plot(h['k1'], label=f'α = {a}', lw=2 if a == ALFA else 1)
axes[0].axhline(1, color='black', lw=0.8)
axes[0].set_title('r = aporte imagen / aporte texto (1 = equilibrio)', fontsize=10)
axes[1].set_title('coeficiente k del texto (1 = sin frenar)', fontsize=10)
for ax in axes:
    ax.set_xlabel('época'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
t0 = time.time()
for s in SEMILLAS:
    for nombre, ge in [('ogm', False), ('ogm_ge', True)]:
        modelo, _ = entrenar_ogm(tens(Xtr), Ytr, tens(Xdev), Ydev_all, CFG, s, ALFA, ge=ge)
        MODELOS[(nombre, s)] = modelo
        TEMP[f'{nombre}_s{s}'] = ajustar_temperatura(YC['dev_mmimdb'], predecir_logits(modelo, tens(Xdev))[:, COLS])
MODALIDADES = {**art['modalidades'], 'ogm': MODS, 'ogm_ge': MODS}
print(f"{2 * len(SEMILLAS)} modelos OGM entrenados en {time.time() - t0:.0f} s")

## 3. ¿Hay más equilibrio?

En test de MM-IMDB, para cada fusión: AP completo, AP al apagar cada modalidad (vector cero = media de train) e
**índice de dependencia de la imagen** = caída al apagar la imagen / (suma de las dos caídas), como en el 06.

In [ ]:
Xte = conjuntos['test_mmimdb'][0]
filas = []
for n in ['fusion_ingenua', 'ogm', 'ogm_ge']:
    for s in SEMILLAS:
        ap = {k: ap_macro(YC['test_mmimdb'], predecir_logits(MODELOS[(n, s)], tens(Xte, k))[:, COLS])
              for k in [None, 'img', 'txt']}
        ci, ct = ap[None] - ap['img'], ap[None] - ap['txt']
        filas.append({'modelo': n, 'semilla': s, 'ap': ap[None], 'ap_solo_texto': ap['img'], 'ap_solo_imagen': ap['txt'],
                      'caida_sin_imagen': ci, 'caida_sin_texto': ct, 'indice_dependencia_imagen': ci / (ci + ct)})
equilibrio = pd.DataFrame(filas)
fmt = lambda x: '{:.4f} ± {:.4f}'.format(*ic95(x))
equilibrio.groupby('modelo').agg({c: fmt for c in equilibrio.columns if c not in ['modelo', 'semilla']}).T

`ap_solo_imagen` es la fusión con el texto apagado: mide cuánto aprendió la rama de imagen **dentro** de la fusión,
que es lo que OGM-GE intenta mejorar.

## 4. Tabla final: todos los métodos en los cuatro grupos

Logits calibrados (temperatura de dev) de cada modelo y semilla; las compuertas del C1 y C2 se aplican sin
cambios. Primero media ± IC sobre semillas; después bootstrap por películas con los logits promedio de las 5
semillas, como en el C1.

In [ ]:
Z = {}
for c, (x, _) in conjuntos.items():
    for (n, s), modelo in MODELOS.items():
        Z[(c, n, s)] = predecir_logits(modelo, [torch.from_numpy(x[m]).to(device) for m in MODALIDADES[n]])[:, COLS] / TEMP[f'{n}_s{s}']

def salida(met, c, s):
    if met == 'ingenua':
        return Z[(c, 'fusion_ingenua', s)]
    if met in ('ogm', 'ogm_ge'):
        return Z[(c, met, s)]
    return combinar(COMPUERTAS[(met, s)], np.stack([Z[(c, n, s)] for n in ENTRADAS[met]], axis=-1))[0]

SAL = {(met, c, s): salida(met, c, s) for met in METODOS for c in GRUPOS for s in SEMILLAS}
filas = [{'grupo': c, 'metodo': met, 'semilla': s, 'ap_macro': ap_macro(YC[c], SAL[(met, c, s)]),
          'logloss': bce_por_etiqueta(YC[c], SAL[(met, c, s)]).sum(1).mean()}
         for met in METODOS for c in GRUPOS for s in SEMILLAS]
por_semilla = pd.DataFrame(filas)
tabla = por_semilla.pivot_table(index='metodo', columns='grupo', values='ap_macro',
                                aggfunc=lambda x: '{:.2f} ± {:.2f}'.format(*ic95(100 * x))).loc[METODOS, GRUPOS]
tabla.insert(0, 'notebook', [ORIGEN[m] for m in METODOS])
print("AP macro (puntos, media ± IC 95% sobre semillas)")
tabla

In [ ]:
ZP = {(met, c): np.mean([SAL[(met, c, s)] for s in SEMILLAS], axis=0) for met in METODOS for c in GRUPOS}
im = {m: k for k, m in enumerate(METODOS)}

def evaluar(c, r):
    return np.array([[ap_macro(YC[c][r], ZP[(met, c)][r]), bce_por_etiqueta(YC[c][r], ZP[(met, c)][r]).sum(1).mean()]
                     for met in METODOS])

rng = np.random.default_rng(SEED)
t0 = time.time()
B = {c: np.array([evaluar(c, rng.integers(0, len(YC[c]), len(YC[c]))) for _ in range(N_BOOT)]) for c in GRUPOS}
PUNTO = {c: evaluar(c, slice(None)) for c in GRUPOS}
print(f"Bootstrap: {time.time() - t0:.0f} s")

filas = []
for j, medida in enumerate(['ap_macro', 'logloss']):
    for met in METODOS[1:]:
        for c in GRUPOS:
            d = B[c][:, im[met], j] - B[c][:, 0, j]
            filas.append({'metodo': met, 'grupo': c, 'medida': medida, 'diferencia': PUNTO[c][im[met], j] - PUNTO[c][0, j],
                          'ic_inf': np.percentile(d, 2.5), 'ic_sup': np.percentile(d, 97.5)})
        g = lambda arr: arr[..., im[met], j] - arr[..., 0, j]
        d = g(B['recientes_tmdb']) - g(B['control_tmdb'])
        filas.append({'metodo': met, 'grupo': 'dif_en_dif', 'medida': medida,
                      'diferencia': g(PUNTO['recientes_tmdb']) - g(PUNTO['control_tmdb']),
                      'ic_inf': np.percentile(d, 2.5), 'ic_sup': np.percentile(d, 97.5)})
boot = pd.DataFrame(filas)
boot['sin_cero'] = (boot['ic_inf'] > 0) | (boot['ic_sup'] < 0)
v = boot[boot.medida == 'ap_macro'].copy()
v['texto'] = v.apply(lambda r: f"{100 * r.diferencia:+.2f} [{100 * r.ic_inf:+.2f}, {100 * r.ic_sup:+.2f}]{' *' if r.sin_cero else ''}", axis=1)
print("AP macro − ingenua, puntos (bootstrap 95% por películas; * = sin el 0)")
v.pivot(index='metodo', columns='grupo', values='texto').loc[METODOS[1:], GRUPOS + ['dif_en_dif']]

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(GRUPOS))
otros = METODOS[1:]
for k, met in enumerate(otros):
    d = boot[(boot.metodo == met) & (boot.medida == 'ap_macro')].set_index('grupo').loc[GRUPOS]
    off = (k - (len(otros) - 1) / 2) * 0.09
    ax.errorbar(x + off, 100 * d['diferencia'], yerr=[100 * (d['diferencia'] - d['ic_inf']), 100 * (d['ic_sup'] - d['diferencia'])],
                fmt='o', ms=4, capsize=2, color=COL[met], label=f'{met} ({ORIGEN[met]})')
ax.axhline(0, color='black', lw=0.8)
ax.axhline(-100 * MARGEN_NO_INFERIORIDAD, color='gray', lw=0.8, ls=':')
ax.set_xticks(x, [g.replace('_', '\n') for g in GRUPOS], fontsize=8)
ax.set_ylabel('AP macro − ingenua (puntos)'); ax.legend(fontsize=7, ncol=4)
ax.set_title('Todos los métodos contra la fusión ingenua (bootstrap 95% por películas)')
plt.tight_layout(); plt.show()

## 5. Análisis de fallos: ¿dónde y cómo se equivoca el texto con las recientes?

**Por género:** cambio de AP de la época (`recientes_tmdb` − `control_tmdb`) en cada rama, en la fusión ingenua y
en `ogm_ge`, con los logits promedio de las semillas. Si los géneros que más pierden son los mismos para el texto
y para la fusión, la fusión hereda el fallo del texto.

In [ ]:
def ap_genero(c, z):
    return ap_por_clase_rapido(YC[c], z)

Zm = lambda c, n: np.mean([Z[(c, n, s)] for s in SEMILLAS], axis=0)
por_genero = pd.DataFrame({
    'prev_control': YC['control_tmdb'].mean(0), 'prev_recientes': YC['recientes_tmdb'].mean(0),
    **{f'cambio_{n}': ap_genero('recientes_tmdb', Zm('recientes_tmdb', n)) - ap_genero('control_tmdb', Zm('control_tmdb', n))
       for n in ['imagen', 'texto', 'fusion_ingenua', 'ogm_ge']}}, index=GENEROS)
print(f"Correlación entre géneros, cambio del texto vs cambio de la fusión: "
      f"r = {np.corrcoef(por_genero['cambio_texto'], por_genero['cambio_fusion_ingenua'])[0, 1]:.2f}")
por_genero.sort_values('cambio_texto').round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
d = por_genero.sort_values('cambio_texto')
x = np.arange(len(d))
for k, (n, c) in enumerate([('imagen', '#f59e0b'), ('texto', '#2563eb'), ('fusion_ingenua', '#10b981'), ('ogm_ge', COL['ogm_ge'])]):
    ax.bar(x + (k - 1.5) * 0.2, 100 * d[f'cambio_{n}'], width=0.2, color=c, label=n)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x, d.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('AP recientes − control (puntos)'); ax.legend(fontsize=8)
ax.set_title('Efecto de la época por género')
plt.tight_layout(); plt.show()

**Por película:** las recientes donde el texto se equivoca **con más seguridad**: mayor log-loss del texto entre las
que tienen la entropía del texto por debajo de la mediana. Se muestran los géneros reales y los tres más probables
según el texto y según la imagen.

In [ ]:
zt, zi = Zm('recientes_tmdb', 'texto'), Zm('recientes_tmdb', 'imagen')
Yr = YC['recientes_tmdb']
L_txt, H_txt = bce_por_etiqueta(Yr, zt).sum(1), entropia_binaria(zt).sum(1)
L_img = bce_por_etiqueta(Yr, zi).sum(1)
seguras = np.where(H_txt <= np.median(H_txt))[0]
peores = seguras[np.argsort(-L_txt[seguras])[:12]]
top3 = lambda z: ', '.join(np.array(GENEROS)[np.argsort(-z)[:3]])
meta_rec = tm.iloc[i_rec].reset_index(drop=True)
col_tit = 'titulo' if 'titulo' in meta_rec else 'tconst'
fallos = pd.DataFrame({
    'titulo': meta_rec[col_tit].values[peores],
    'anio': meta_rec['anio'].values[peores] if 'anio' in meta_rec else np.nan,
    'generos_reales': [', '.join(np.array(GENEROS)[Yr[i] > 0]) for i in peores],
    'texto_dice': [top3(zt[i]) for i in peores],
    'imagen_dice': [top3(zi[i]) for i in peores],
    'logloss_texto': L_txt[peores].round(2), 'logloss_imagen': L_img[peores].round(2),
    'entropia_texto': H_txt[peores].round(2)})
print(f"En las recientes, la imagen tiene menor log-loss que el texto en {np.mean(L_img < L_txt):.0%} de las películas "
      f"({np.mean(L_img[seguras] < L_txt[seguras]):.0%} entre las que el texto está seguro)")
fallos

## 6. Criterios

In [ ]:
def fb(met, grupo):
    return boot[(boot.metodo == met) & (boot.grupo == grupo) & (boot.medida == 'ap_macro')].iloc[0]

e = equilibrio.pivot(index='semilla', columns='modelo', values='indice_dependencia_imagen')
m_, h_ = ic95(e['ogm_ge'] - e['fusion_ingenua'])
c2, c3, c4 = fb('ogm_ge', 'test_mmimdb'), fb('ogm_ge', 'recientes_tmdb'), fb('ogm_ge', 'dif_en_dif')
pts = lambda r: f"{100 * r.diferencia:+.2f} [{100 * r.ic_inf:+.2f}, {100 * r.ic_sup:+.2f}] pts"
criterios = pd.DataFrame([
    {'criterio': '1. ogm_ge: más dependencia de la imagen que la ingenua', 'valor': f'{m_:+.3f} ± {h_:.3f}', 'cumple': m_ - h_ > 0},
    {'criterio': '2. ogm_ge no inferior en test_mmimdb', 'valor': pts(c2), 'cumple': c2.ic_inf > -MARGEN_NO_INFERIORIDAD},
    {'criterio': '3. ogm_ge mejora en recientes_tmdb', 'valor': pts(c3), 'cumple': c3.ic_inf > 0},
    {'criterio': '4. ganancia mayor en recientes que en control', 'valor': pts(c4), 'cumple': c4.ic_inf > 0},
])
criterios

## Guardado

- `exp6_busqueda_alfa.csv`, `exp6_equilibrio.csv`: elección de α e índice de dependencia.
- `exp6_por_semilla.csv`, `exp6_bootstrap.csv`, `exp6_tabla_final.csv`: la tabla final de todos los métodos.
- `exp6_por_genero.csv`, `exp6_fallos.csv`: análisis de fallos.
- `exp6_criterios.csv`, `exp6_config.json`.
- `exp6_ogm_modelos.pt`: pesos y temperaturas de `ogm` y `ogm_ge` (misma estructura de `Clasificador`).

In [ ]:
busqueda.to_csv(f'{RES}/exp6_busqueda_alfa.csv', index=False)
equilibrio.to_csv(f'{RES}/exp6_equilibrio.csv', index=False)
por_semilla.to_csv(f'{RES}/exp6_por_semilla.csv', index=False)
boot.to_csv(f'{RES}/exp6_bootstrap.csv', index=False)
tabla.to_csv(f'{RES}/exp6_tabla_final.csv')
por_genero.to_csv(f'{RES}/exp6_por_genero.csv')
fallos.to_csv(f'{RES}/exp6_fallos.csv', index=False)
criterios.to_csv(f'{RES}/exp6_criterios.csv', index=False)
torch.save({'alfa': ALFA, 'config': CFG, 'generos_entrenamiento': G_ENT, 'columnas_eval': GENEROS,
            'modelos': {f'{n}_s{s}': {k: v.cpu() for k, v in MODELOS[(n, s)].state_dict().items()}
                        for n in ['ogm', 'ogm_ge'] for s in SEMILLAS},
            'temperaturas': {f'{n}_s{s}': TEMP[f'{n}_s{s}'] for n in ['ogm', 'ogm_ge'] for s in SEMILLAS}},
           f'{RES}/exp6_ogm_modelos.pt')
with open(f'{RES}/exp6_config.json', 'w') as fh:
    json.dump({'generos': GENEROS, 'semillas': SEMILLAS, 'alfas': ALFAS, 'alfa_elegido': ALFA, 'n_bootstrap': N_BOOT,
               'semilla_bootstrap': SEED, 'margen_no_inferioridad': MARGEN_NO_INFERIORIDAD, 'metodos': METODOS,
               'origen': ORIGEN}, fh, indent=2)
print(sorted(f for f in os.listdir(RES) if f.startswith('exp6')))